Feature Engineering und unterschiedliche Nachbarschaftkodierungen

In [5]:
import numpy as np
import pandas as pd

print("--- 1. LADE BEREINIGTE DATEN & ERSTELLE BASIS-DATENSATZ (NO GEO) ---")

# Bereinigten Datensatz laden
df_iqr_cut_adj = pd.read_pickle("df_iqr_cut_adj.pkl")
df_features = df_iqr_cut_adj.copy()
print(f"Geladene Datenbasis: {df_features.shape}")

# ==============================================================================
# BEREINIGUNG & STRUKTURELLE MERKMALE
# ==============================================================================
def parse_amenities_count(x):
    if pd.isna(x) or not isinstance(x, str): return 0
    clean_str = x.strip()
    if clean_str in ["[]", "{}", ""]: return 0
    items = [item.strip() for item in clean_str[1:-1].split(",") if item.strip()]
    return len(items)

df_features["amenity_count"] = (
    df_features["amenities"].apply(parse_amenities_count) 
    if "amenities" in df_features.columns else 0
)

for rate_col in ["host_response_rate", "host_acceptance_rate"]:
    if rate_col in df_features.columns:
        df_features[rate_col] = df_features[rate_col].astype(str).str.replace("%", "").astype(float) / 100.0
        df_features[rate_col] = df_features[rate_col].fillna(df_features[rate_col].median())
    else:
        df_features[rate_col] = 1.0

for cap_col in ["bedrooms", "beds", "bathrooms"]:
    if cap_col in df_features.columns:
        df_features[cap_col] = df_features[cap_col].fillna(df_features[cap_col].median())

# ==============================================================================
# RATIOS & METRIKEN
# ==============================================================================
safe_bedrooms = df_features["bedrooms"].replace(0, 1)
safe_price = df_features["price"].replace(0, 1)
safe_accommodates = df_features["accommodates"].replace(0, 1)

df_features["beds_per_bedroom"] = df_features["beds"] / safe_bedrooms
df_features["guests_per_bedroom"] = df_features["accommodates"] / safe_bedrooms
df_features["amenities_per_euro"] = df_features["amenity_count"] / safe_price
df_features["bathrooms_per_guest"] = df_features["bathrooms"] / safe_accommodates

if "bathrooms_text" in df_features.columns:
    df_features["is_shared_bathroom"] = df_features["bathrooms_text"].fillna("").astype(str).str.lower().str.contains("shared").astype(int)
else:
    df_features["is_shared_bathroom"] = 0

if "host_response_time" in df_features.columns:
    response_mapping = {"within an hour": 4, "within a few hours": 3, "within a day": 2, "a few days or more": 1}
    df_features["host_response_score"] = df_features["host_response_time"].map(response_mapping).fillna(3)
else:
    df_features["host_response_score"] = 3

if "host_since" in df_features.columns:
    df_features["host_since_dt"] = pd.to_datetime(df_features["host_since"], errors="coerce")
    df_features["host_since_dt"] = df_features["host_since_dt"].fillna(df_features["host_since_dt"].median())
    df_features["host_seniority_years"] = 2025 - df_features["host_since_dt"].dt.year
else:
    df_features["host_seniority_years"] = 0

df_features["availability_ratio_30"] = df_features["availability_30"] / 30.0 if "availability_30" in df_features.columns else 0
df_features["min_nights_to_availability_ratio"] = df_features["minimum_nights"] / (df_features["availability_365"] + 1)
df_features["description_length"] = df_features["description"].fillna("").astype(str).str.len() if "description" in df_features.columns else 0

if "host_identity_verified" in df_features.columns:
    df_features["is_identity_verified"] = df_features["host_identity_verified"].map({"t": 1, "f": 0}).fillna(0)
else:
    df_features["is_identity_verified"] = 0

df_features["price_per_guest"] = df_features["price"] / safe_accommodates

# ==============================================================================
# BASIS-DATAFRAME ZUSAMMENSTELLEN & EXPORTIEREN
# ==============================================================================
base_features = [
    "accommodates", "price", "minimum_nights", "calculated_host_listings_count", "availability_365",
    "is_superhost", "is_instant_bookable", "amenity_count", "host_response_rate", "host_acceptance_rate",
    "bedrooms", "beds", "bathrooms", "beds_per_bedroom", "guests_per_bedroom", "amenities_per_euro",
    "bathrooms_per_guest", "is_shared_bathroom", "host_response_score", "host_seniority_years",
    "availability_ratio_30", "min_nights_to_availability_ratio", "description_length",
    "is_identity_verified", "price_per_guest"
]

cols_to_keep = base_features + ["room_type", "review_scores_rating"]

df_model_base_no_geo = pd.get_dummies(
    df_features[cols_to_keep], 
    columns=["room_type"], 
    dtype=int
)

# Export für Folge-Notebooks
df_model_base_no_geo.to_pickle("df_model_base_no_geo.pkl")
print(f"df_model_base_no_geo bereit & gespeichert: {df_model_base_no_geo.shape}")

--- 1. LADE BEREINIGTE DATEN & ERSTELLE BASIS-DATENSATZ (NO GEO) ---
Geladene Datenbasis: (6298, 38)
df_model_base_no_geo bereit & gespeichert: (6298, 30)


In [9]:
print("--- 2. ERWEITERE BASIS UM ONE-HOT NACHBARSCHAFTEN ---")

# Dummies der Kiez-Spalte direkt aus df_iqr_cut_adj bilden
ohe_neighbourhoods = pd.get_dummies(
    df_iqr_cut_adj["neighbourhood_cleansed"],
    prefix="neighbourhood_cleansed",
    dtype=int,
)

# Anzahl der Nachbarschaften ermitteln und ausgeben
num_neighbourhoods = ohe_neighbourhoods.shape[1]
print(f"Anzahl generierter Dummy-Variablen (Nachbarschaften): {num_neighbourhoods}")
print(f"Anzahl einzigartiger Werte in 'neighbourhood_cleansed': {df_iqr_cut_adj['neighbourhood_cleansed'].nunique()}")

# Direkt an den Basis-Datensatz anhängen
df_model_neighbourhood_ohe = pd.concat(
    [df_model_base_no_geo, ohe_neighbourhoods], axis=1
)

# Export für Folge-Notebooks
df_model_neighbourhood_ohe.to_pickle("df_model_neighbourhood_ohe.pkl")
print(f"df_model_neighbourhood_ohe bereit & gespeichert: {df_model_neighbourhood_ohe.shape}")

--- 2. ERWEITERE BASIS UM ONE-HOT NACHBARSCHAFTEN ---
Anzahl generierter Dummy-Variablen (Nachbarschaften): 135
Anzahl einzigartiger Werte in 'neighbourhood_cleansed': 135
df_model_neighbourhood_ohe bereit & gespeichert: (6298, 165)


In [8]:
from sklearn.model_selection import KFold

print("--- 3. ERWEITERE BASIS UM LEAKAGE-FREIES TARGET-ENCODING ---")

# 5-Fold OOF-Berechnung auf df_iqr_cut_adj
kf = KFold(n_splits=5, shuffle=True, random_state=42)
kiez_encoded = pd.Series(index=df_iqr_cut_adj.index, dtype=float)
global_median = df_iqr_cut_adj["review_scores_rating"].median()

for train_idx, val_idx in kf.split(df_iqr_cut_adj):
    train_data = df_iqr_cut_adj.iloc[train_idx]
    fold_kiez_medians = train_data.groupby("neighbourhood_cleansed")["review_scores_rating"].median()
    kiez_encoded.iloc[val_idx] = df_iqr_cut_adj.iloc[val_idx]["neighbourhood_cleansed"].map(fold_kiez_medians)

# An die Basis anhängen
df_model_target_encoded = df_model_base_no_geo.copy()
df_model_target_encoded["kiez_encoded_rating_oof"] = kiez_encoded.fillna(global_median)

# Export für Folge-Notebooks
df_model_target_encoded.to_pickle("df_model_target_encoded.pkl")
print(f"df_model_target_encoded bereit & gespeichert: {df_model_target_encoded.shape}")

--- 3. ERWEITERE BASIS UM LEAKAGE-FREIES TARGET-ENCODING ---
df_model_target_encoded bereit & gespeichert: (6298, 31)
